In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Cargar dataset de actividades
actividades = pd.read_csv("actividades.csv")

# Vectorizar descripciones o intereses
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(actividades['descripcion_larga'] + ' ' + actividades['nombre'])

def recomendar(intereses_usuario, edad, entorno, coste_max):
    user_vector = vectorizer.transform([intereses_usuario])
    similitudes = cosine_similarity(user_vector, X).flatten()

    actividades['score'] = similitudes
    recomendaciones = actividades[
        (actividades['edad_minima'] <= edad) &
        (actividades['edad_maxima'] >= edad) &
        (actividades['id_entorno'].astype(str).str.contains(entorno.lower())) &
        (actividades['id_coste'] <= coste_max)
    ].sort_values(by='score', ascending=False)

    return recomendaciones[['nombre', 'descripcion_corta', 'score']].head(5).to_dict(orient='records')


In [2]:
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

app = FastAPI()

# CORS para permitir peticiones desde el frontend local
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Cargar y preparar datos
df = pd.read_csv("actividades.csv")
vectorizer = TfidfVectorizer()
matriz_tfidf = vectorizer.fit_transform(df['descripcion_larga'] + ' ' + df['nombre'])

# Modelo de entrada
class EntradaUsuario(BaseModel):
    edad: int
    intereses: str
    ubicacion: str
    coste: float
    entorno: str
    temporada: str

@app.post("/recomendar")
def recomendar(input: EntradaUsuario):
    # Vectorizar intereses del usuario
    intereses_vector = vectorizer.transform([input.intereses])
    similitudes = cosine_similarity(intereses_vector, matriz_tfidf).flatten()
    df['score'] = similitudes

    # Filtrar actividades compatibles
    filtradas = df[
        (df['edad_minima'] <= input.edad) &
        (df['edad_maxima'] >= input.edad) &
        (df['id_entorno'].str.lower() == input.entorno.lower()) &
        (df['id_coste'] <= input.coste)
    ]

    recomendaciones = filtradas.sort_values(by='score', ascending=False).head(5)
    return recomendaciones[['nombre', 'descripcion_corta', 'score']].to_dict(orient='records')